# Text analytics with NLTK and sentence transformers

Four small experiments:

- **a)** language detection by counting stop words
- **b)** movie title search with character n-grams (2 to 4 letters) and Jaccard similarity
- **c)** the same search with sentence transformers
- **d)** language detection with Naive Bayes on character n-grams, compared with a) on short texts

Data: IMDB top 1000 movies (Kaggle) and the multilingual European Parliament texts in NLTK (`europarl_raw`).

In [1]:
import re
import math
from collections import Counter

import nltk
import numpy as np
import pandas as pd
from unidecode import unidecode

nltk.download("stopwords", quiet=True)
nltk.download("europarl_raw", quiet=True)
from nltk.corpus import stopwords, europarl_raw

## a) Language detection with stop words

Count how many words of the text are in the NLTK stop word list of each language and pick the language with the most hits.

In [2]:
LANGS = ["english", "german", "italian"]
STOP = {lang: set(stopwords.words(lang)) for lang in LANGS}

def words(text):
    return re.findall(r"\w+", text.lower())

def detect_by_stopwords(text, langs=LANGS):
    counts = {lang: sum(w in STOP[lang] for w in words(text)) for lang in langs}
    return max(counts, key=counts.get), counts

Test texts: the first 1000 characters of a parliament session in each language.

In [3]:
def corpus(lang):
    return getattr(europarl_raw, lang)

samples = {lang: corpus(lang).raw(corpus(lang).fileids()[0])[:1000] for lang in LANGS}

for lang, text in samples.items():
    pred, counts = detect_by_stopwords(text)
    print(f"{lang:8} -> {pred:8} {counts}")

english  -> english  {'english': 89, 'german': 7, 'italian': 19}
german   -> german   {'english': 4, 'german': 64, 'italian': 3}
italian  -> italian  {'english': 7, 'german': 4, 'italian': 69}


It works well on long texts. On a few words it often has nothing to count:

In [4]:
for text in ["Guten Morgen", "Grazie mille", "Thank you very much", "Das Wetter ist gut", "Buona notte"]:
    pred, counts = detect_by_stopwords(text)
    print(f"{text:22} -> {pred:8} {counts}")

Guten Morgen           -> english  {'english': 0, 'german': 0, 'italian': 0}
Grazie mille           -> english  {'english': 0, 'german': 0, 'italian': 0}
Thank you very much    -> english  {'english': 2, 'german': 0, 'italian': 0}
Das Wetter ist gut     -> german   {'english': 0, 'german': 2, 'italian': 0}
Buona notte            -> english  {'english': 0, 'german': 0, 'italian': 0}


## b) Title search with character n-grams

Each word is padded with `^` and `$` so that the start and end of a word give their own n-grams, and accents are removed with `unidecode`. A title becomes a set of 2-, 3- and 4-grams. The similarity between query and title is the Jaccard index |A ∩ B| / |A ∪ B|.

In [5]:
movies = pd.read_csv("data/imdb_top_1000.csv")
titles = movies["Series_Title"].tolist()
print(len(titles), "titles, e.g.", titles[:5])

1000 titles, e.g. ['The Shawshank Redemption', 'The Godfather', 'The Dark Knight', 'The Godfather: Part II', '12 Angry Men']


In [6]:
def ngrams(text, sizes=(2, 3, 4)):
    text = unidecode(text).lower()
    grams = set()
    for word in re.findall(r"[a-z0-9]+", text):
        word = "^" + word + "$"
        for n in sizes:
            for i in range(len(word) - n + 1):
                grams.add(word[i:i + n])
    return grams

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

title_grams = [ngrams(t) for t in titles]

def search_ngrams(query, k=5):
    q = ngrams(query)
    scores = [jaccard(q, g) for g in title_grams]
    best = np.argsort(scores)[::-1][:k]
    return [(titles[i], round(scores[i], 3)) for i in best]

print(sorted(ngrams("Amélie"))[:10])

['^a', '^am', '^ame', 'am', 'ame', 'amel', 'e$', 'el', 'eli', 'elie']


In [7]:
queries = ["godfather", "lord of the rings", "the dark night", "shawshenk", "amelie", "space adventure"]
for q in queries:
    print(q, "->", search_ngrams(q, k=3))

godfather -> [('The Godfather', 0.818), ('The Godfather: Part II', 0.529), ('The Godfather: Part III', 0.509)]
lord of the rings -> [('The Lord of the Rings: The Two Towers', 0.636), ('The Lord of the Rings: The Return of the King', 0.618), ('The Lord of the Rings: The Fellowship of the Ring', 0.568)]
the dark night -> [('The Dark Knight', 0.786), ('The Dark Knight Rises', 0.579), ('A Night at the Opera', 0.421)]
shawshenk -> [('The Shawshank Redemption', 0.254), ('Sherlock Jr.', 0.12), ('Shaun of the Dead', 0.115)]
amelie -> [('Amélie', 1.0), ('In America', 0.154), ('Alien', 0.138)]
space adventure -> [('The Adventures of Robin Hood', 0.284), ('Office Space', 0.268), ("L'avventura", 0.224)]


Spelling mistakes ("shawshenk", "the dark night") and missing accents ("amelie") are no problem, because most n-grams still match. But a query about the content ("space adventure") only finds titles that share letters.

## c) Title search with sentence transformers

Titles and queries are embedded and compared with the dot product of normalized vectors (= cosine similarity). I compare a small and a larger model.

In [8]:
from sentence_transformers import SentenceTransformer

MODELS = ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]
encoders, title_vecs = {}, {}
for name in MODELS:
    encoders[name] = SentenceTransformer(name, device="cpu")
    title_vecs[name] = encoders[name].encode(titles, normalize_embeddings=True, batch_size=64)
    print(name, title_vecs[name].shape)

def search_semantic(query, model="all-MiniLM-L6-v2", k=3):
    q = encoders[model].encode(query, normalize_embeddings=True)
    scores = title_vecs[model] @ q
    best = np.argsort(scores)[::-1][:k]
    return [(titles[i], round(float(scores[i]), 3)) for i in best]

/Users/matinroozbahani/rl-mdp-to-llm-alignment/.venv-arm/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


all-MiniLM-L6-v2 (1000, 384)


all-mpnet-base-v2 (1000, 768)


In [9]:
queries = ["godfather", "the dark night", "shawshenk", "space adventure", "a boy who is a wizard", "toys that come alive"]
for q in queries:
    print(q)
    print("   n-grams :", search_ngrams(q, k=3))
    for name in MODELS:
        print(f"   {name:17}:", search_semantic(q, name))

godfather
   n-grams : [('The Godfather', 0.818), ('The Godfather: Part II', 0.529), ('The Godfather: Part III', 0.509)]
   all-MiniLM-L6-v2 : [('The Godfather', 0.966), ('The Godfather: Part II', 0.864), ('The Godfather: Part III', 0.848)]


   all-mpnet-base-v2: [('The Godfather', 0.885), ('The Godfather: Part II', 0.8), ('The Godfather: Part III', 0.792)]
the dark night
   n-grams : [('The Dark Knight', 0.786), ('The Dark Knight Rises', 0.579), ('A Night at the Opera', 0.421)]
   all-MiniLM-L6-v2 : [('Night on Earth', 0.721), ('Dark City', 0.703), ('Dark Waters', 0.664)]
   all-mpnet-base-v2: [('Dancer in the Dark', 0.697), ("A Hard Day's Night", 0.677), ('Dark City', 0.663)]
shawshenk
   n-grams : [('The Shawshank Redemption', 0.254), ('Sherlock Jr.', 0.12), ('Shaun of the Dead', 0.115)]
   all-MiniLM-L6-v2 : [('Zwartboek', 0.514), ('Barton Fink', 0.473), ('The Shawshank Redemption', 0.444)]


   all-mpnet-base-v2: [('Zwartboek', 0.476), ('Eskiya', 0.461), ('Forushande', 0.446)]
space adventure
   n-grams : [('The Adventures of Robin Hood', 0.284), ('Office Space', 0.268), ("L'avventura", 0.224)]
   all-MiniLM-L6-v2 : [('Star Trek', 0.589), ('Star Wars', 0.541), ('2001: A Space Odyssey', 0.539)]
   all-mpnet-base-v2: [('2001: A Space Odyssey', 0.603), ('Star Trek', 0.55), ('Star Trek Into Darkness', 0.539)]
a boy who is a wizard
   n-grams : [('The Wizard of Oz', 0.281), ('A Star Is Born', 0.203), ('Gully Boy', 0.155)]
   all-MiniLM-L6-v2 : [('The Boy in the Striped Pyjamas', 0.516), ('The Wizard of Oz', 0.469), ('The Kid', 0.466)]


   all-mpnet-base-v2: [('The Wizard of Oz', 0.488), ('The Boy in the Striped Pyjamas', 0.474), ('Harry Potter and the Goblet of Fire', 0.465)]
toys that come alive
   n-grams : [('All That Jazz', 0.224), ('The King of Comedy', 0.16), ('Home Alone', 0.136)]
   all-MiniLM-L6-v2 : [('Toy Story', 0.629), ('Toy Story 3', 0.588), ('Toy Story 4', 0.574)]
   all-mpnet-base-v2: [('Toy Story', 0.474), ('Toy Story 3', 0.457), ('Toy Story 2', 0.443)]


The two methods fail in opposite cases:

- **Typos:** n-grams find "The Dark Knight" for "the dark night" and "The Shawshank Redemption" for "shawshenk". The embeddings miss both, because a misspelled word is split into different tokens and gets a different meaning ("night" instead of "knight").
- **Meaning:** for "space adventure" and "toys that come alive" the embeddings find Star Trek, 2001: A Space Odyssey and Toy Story, while n-grams only match letters ("Office Space", "All That Jazz").

The embeddings only see the title, not the plot, so a description like "a boy who is a wizard" works only partly (Harry Potter is 3rd with the larger model). The larger model is not clearly better here.

## d) Naive Bayes language detection on character n-grams

Five languages with the same alphabet (after `unidecode`). For each language I count all 2- to 4-grams in 8 of the 10 parliament sessions and keep the top n = 1000. The likelihood of an n-gram is its count divided by the total count of the kept n-grams, and the prior is equal for all languages, so

score(lang) = Σ log P(gram | lang) over the n-grams of the text.

N-grams that are in no language's table are ignored. If an n-gram is in the table of some languages but not in another, that language gets a small floor probability for it instead of 0 (otherwise one missing n-gram would make the whole posterior 0).

In [10]:
NB_LANGS = ["english", "german", "italian", "french", "spanish"]
TOP_N = 1000
for lang in NB_LANGS:   # stop word lists for the comparison with a)
    STOP[lang] = set(stopwords.words(lang))

def ngram_counts(text, sizes=(2, 3, 4)):
    text = unidecode(text).lower()
    counts = Counter()
    for word in re.findall(r"[a-z]+", text):
        word = "^" + word + "$"
        for n in sizes:
            for i in range(len(word) - n + 1):
                counts[word[i:i + n]] += 1
    return counts

def split_sessions(lang):
    files = corpus(lang).fileids()
    return files[:8], files[8:]

likelihood, floor = {}, {}
for lang in NB_LANGS:
    train_files, _ = split_sessions(lang)
    counts = Counter()
    for f in train_files:
        counts.update(ngram_counts(corpus(lang).raw(f)))
    top = dict(counts.most_common(TOP_N))
    total = sum(top.values())
    likelihood[lang] = {g: c / total for g, c in top.items()}
    floor[lang] = min(likelihood[lang].values()) / 2
    print(lang, "top 5:", list(top)[:5])

known = set().union(*likelihood.values())

def detect_by_nb(text):
    grams = ngram_counts(text)
    scores = {}
    for lang in NB_LANGS:
        s = 0.0
        for g, c in grams.items():
            if g in known:
                s += c * math.log(likelihood[lang].get(g, floor[lang]))
        scores[lang] = s
    return max(scores, key=scores.get)

english top 5: ['e$', '^t', 'th', 's$', '^th']


german top 5: ['n$', 'en', 'er', 'en$', '^d']


italian top 5: ['e$', 'a$', 'i$', 'o$', '^d']


french top 5: ['e$', 's$', '^d', 'es', 't$']


spanish top 5: ['e$', 'a$', 's$', 'o$', '^e']


In [11]:
for text in ["Guten Morgen", "Grazie mille", "Thank you very much", "Das Wetter ist gut", "Buona notte", "Merci beaucoup", "Muchas gracias"]:
    print(f"{text:22} -> NB: {detect_by_nb(text):8} stop words: {detect_by_stopwords(text, NB_LANGS)[0]}")

Guten Morgen           -> NB: german   stop words: english
Grazie mille           -> NB: italian  stop words: english
Thank you very much    -> NB: english  stop words: english
Das Wetter ist gut     -> NB: german   stop words: german
Buona notte            -> NB: italian  stop words: english
Merci beaucoup         -> NB: english  stop words: english
Muchas gracias         -> NB: spanish  stop words: english


### Evaluation on held-out sentences

Test sentences come from the 2 sessions not used for training. To see how both methods behave on short texts, every sentence is also cut to its first 1, 3 and 6 words.

In [12]:
rows = []
for lang in NB_LANGS:
    _, test_files = split_sessions(lang)
    # the raw files have one sentence per line
    sents = [line.strip() for f in test_files for line in corpus(lang).raw(f).splitlines()]
    sents = [s for s in sents if len(s.split()) >= 10][:300]
    for s in sents:
        for n_words in [1, 3, 6, None]:
            text = " ".join(s.split()[:n_words]) if n_words else s
            rows.append({
                "lang": lang,
                "length": (f"{n_words} word" + ("s" if n_words > 1 else "")) if n_words else "full sentence",
                "stop words": detect_by_stopwords(text, NB_LANGS)[0] == lang,
                "naive bayes": detect_by_nb(text) == lang,
            })

results = pd.DataFrame(rows)
order = ["1 word", "3 words", "6 words", "full sentence"]
acc = results.groupby("length")[["stop words", "naive bayes"]].mean().loc[order]
print(len(results) // 4, "test sentences")
(acc * 100).round(1)

1500 test sentences


,stop words,naive bayes
length,,
1 word,60.0,75.7
3 words,81.3,94.5
6 words,93.3,98.9
full sentence,99.7,100.0


In [13]:
(results[results["length"] == "3 words"].groupby("lang")[["stop words", "naive bayes"]].mean() * 100).round(1)

,stop words,naive bayes
lang,,
english,100.0,92.7
french,74.3,93.7
german,92.3,99.0
italian,80.3,92.7
spanish,59.7,94.3


Naive Bayes on n-grams is clearly better on short texts: 76% vs. 60% with one word and 95% vs. 81% with three words. On full sentences both are close to 100%.

The stop word method looks perfect for English only because of ties: if a text has no stop words at all, every language has 0 hits and `max` returns the first one, English. That is also why it does badly on Spanish and French, which share many short words with each other. Naive Bayes still makes mistakes on very short phrases, e.g. "Merci beaucoup" comes out as English, because these words are rare in parliament speeches.